In [14]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd

In [15]:
dataset_path = "../../bats_transformer/data/2022_barn_daytime_2secs/splits"
dataset_path = "../../bats_transformer/data/2022_barn_2secs_myca/splits"

In [16]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [17]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": dataset_path,
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=64,
    workers=4,
)

In [18]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [19]:
# preds = []
# for batch in tqdm(train_data):
#     x_t, x_c, y_t, y_c = batch
#     mask = x_t > 0
#     lengths = mask.sum(dim=1)
#     feature_sums = x_c.sum(dim=1)
#     # print(feature_sums.shape)
#     for i, row in enumerate(feature_sums):
#         preds.append((row / lengths[i]))
#     # print(preds)


In [20]:
truths = []
preds = []
errors = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    mask = x_t > 0
    lengths = mask.sum(dim=1)
    feature_sums = x_c.sum(dim=1)
    for i, row in enumerate(feature_sums):
        pred = (row / lengths[i])
        truths.append(y_c[i].numpy()[0])
        preds.append(pred.numpy())
        errors.append((y_c[i] - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 171/171 [01:06<00:00,  2.56it/s]


In [21]:
pd.DataFrame(preds)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.530247,-0.548183,-0.643562,-0.916312,0.337398,0.281385,-0.169823,0.022113,-0.647374,-0.503226,...,1.428009,0.687758,0.206849,0.338262,0.187427,-0.564926,-0.488592,-0.280176,-0.245650,0.167250
1,-0.288188,0.474394,0.048871,0.079668,0.139593,-0.277577,0.151230,-0.709391,0.047365,0.134553,...,-0.438702,-0.063190,0.014389,0.010380,-0.035396,0.245301,0.104836,-0.051966,-0.117194,-0.085631
2,-0.369581,0.467001,0.055773,0.081874,0.137831,-0.256275,0.175044,-0.734511,0.054222,0.123851,...,-0.409883,-0.012037,0.106862,-0.008162,-0.068794,0.282350,0.107625,-0.020362,-0.115546,-0.055815
3,-0.451664,0.428388,0.061333,0.079511,0.123155,-0.209230,0.189128,-0.733869,0.059436,0.154457,...,-0.366957,0.057156,0.057450,0.050631,-0.074131,0.326892,0.128608,0.018757,-0.098355,-0.003265
4,-0.534949,0.435392,0.052613,0.058027,0.081692,-0.145743,0.205738,-0.754894,0.050514,0.189214,...,-0.327851,0.157810,-0.004669,0.093804,-0.087174,0.302720,0.121287,-0.003489,-0.109868,0.095321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10890,-1.080398,0.260315,0.113920,0.197047,-0.025227,-0.129973,-0.289019,0.004223,0.115094,0.300781,...,-0.593292,-0.288247,-0.785402,-0.065235,0.026918,0.301533,0.361973,-0.028560,0.094850,0.075981
10891,-1.228976,-0.057969,0.148814,0.279471,0.029562,-0.268853,-0.359495,0.105236,0.150051,0.422336,...,-0.555489,-0.311525,-0.980380,-0.036678,-0.140089,0.386803,0.399076,-0.025604,0.053868,-0.053876
10892,-1.387736,-0.187765,-0.055709,-0.017799,-0.587483,0.217252,-0.383262,0.141705,-0.054323,0.188070,...,-0.619944,-0.159899,-0.746846,0.129770,0.288660,0.318472,0.210556,-0.099464,-0.090248,0.439637
10893,0.496073,0.933141,-0.869814,-0.818153,-0.751186,0.496543,-1.255367,0.784028,-0.876258,-1.482676,...,0.123099,1.569473,1.007041,0.296035,0.859658,-0.545686,-0.868159,-0.466371,-0.876195,0.681489


In [22]:
target_columns = train_data.dataset.target_cols
# target_columns

In [23]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [24]:
mse

array([1.1695802, 0.9666821, 1.2314947, 1.2819717, 1.3233696, 1.2501405,
       1.2482592, 1.0131986, 1.233498 , 1.3130385, 1.3205389, 1.2275506,
       1.2316844, 9.574592 , 1.1389728, 1.2261907, 1.094919 , 1.3525646,
       1.2708848, 1.1988914, 1.2024492, 1.1958517, 1.4303371, 1.2281067,
       1.2086786, 1.0700616, 1.0286423, 1.4093703, 2.6498466, 1.3841094,
       2.6920485, 1.0782354], dtype=float32)

In [25]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile         1.169580
PrecedingIntrvl    0.966682
HiFreq             1.231495
Bndwdth            1.281972
FreqMaxPwr         1.323370
PrcntMaxAmpDur     1.250141
FreqKnee           1.248259
PrcntKneeDur       1.013199
StartF             1.233498
UpprKnFreq         1.313038
HiFtoUpprKnAmp     1.320539
HiFtoKnAmp         1.227551
HiFtoFcAmp         1.231684
UpprKnToKnAmp      9.574592
KnToFcAmp          1.138973
LdgToFcAmp         1.226191
FreqCtr            1.094919
FFwd32dB           1.352565
FFwd20dB           1.270885
FFwd15dB           1.198891
FBak5dB            1.202449
FFwd5dB            1.195852
Bndw32dB           1.430337
Amp1stQrtl         1.228107
Amp2ndQrtl         1.208679
Amp3rdQrtl         1.070062
Amp4thQrtl         1.028642
1st10kHzSlp        1.409370
1st5to15kHzSlp     2.649847
1st10kHzExp        1.384109
1st5to15kHzExp     2.692049
AmpK@start         1.078235
dtype: float32

In [26]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

1.5701799906249998 1.3119731516129032
